# LeetCode 875 — Koko Eating Bananas bằng Python

Link đề: https://leetcode.com/problems/koko-eating-bananas/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: **tính đơn điệu** — nếu tốc độ `k` đủ để ăn xong trong `h` giờ thì mọi tốc độ lớn hơn cũng đủ. Vì vậy ta **tìm kiếm nhị phân trên đáp án** (binary search on answer). Oracle: **mô phỏng từng giờ đúng như đề** (mỗi giờ ăn tối đa `k` quả từ một đống) rồi quét `k = 1, 2, 3, …`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi giờ Koko chỉ ăn **một đống**, tối đa `k` quả (đống ít hơn `k` thì ăn hết và bỏ phí phần giờ còn lại); tìm `k` nhỏ nhất để xong trong `h` giờ.
2. Nắm công cụ Python: chia nguyên `//`, **chia làm tròn lên** `(p + k - 1) // k` hoặc `-(-p // k)`, `sum` với generator, `max`, `bisect` (có tham số `key` từ Python 3.10).
3. Đi từ **quét tuyến tính `k` (brute force)** → **nhị phân trên `k`** → **nhị phân với cận dưới chặt `⌈tổng/h⌉`** → **`bisect_left` với `key`**.
4. Hiểu mấu chốt: hàm `hours(k) = Σ ⌈pile / k⌉` **không tăng** theo `k`, nên mệnh đề "`hours(k) ≤ h`" có dạng `False … False True … True`; ta tìm `True` đầu tiên.
5. Tránh lỗi kinh điển: dùng `//` thay vì làm tròn lên, cận dưới `0` gây chia cho 0, điều kiện `<` thay vì `≤`, cận trên sai (`sum` thay vì `max`) hoặc vòng nhị phân không kết thúc/lệch một, và tính giờ bằng `⌈tổng/k⌉` (sai vì mỗi giờ chỉ ăn một đống).

## 2. Đề bài & ví dụ

Có `n` đống chuối, đống thứ `i` có `piles[i]` quả. Bảo vệ về sau `h` giờ.

Koko chọn **tốc độ** `k` quả/giờ. Mỗi giờ cô chọn **một** đống và ăn `k` quả từ đó; nếu đống còn ít hơn `k` thì ăn hết và **không ăn thêm** trong giờ đó.

Trả về `k` **nhỏ nhất** để ăn hết chuối **trong vòng `h` giờ**.

**Ví dụ 1:** `piles = [3,6,7,11]`, `h = 8` → `4` (các đống mất `1, 2, 2, 3` giờ, tổng `8`).

**Ví dụ 2:** `piles = [30,11,23,4,20]`, `h = 5` → `30`.

**Ví dụ 3:** `piles = [30,11,23,4,20]`, `h = 6` → `23`.

**Ràng buộc:** `1 <= piles.length <= 10^4`; `piles.length <= h <= 10^9`; `1 <= piles[i] <= 10^9`.

In [1]:
import math
import random
import time
from bisect import bisect_left
from typing import List

examples = [(([3, 6, 7, 11], 8), 4), (([30, 11, 23, 4, 20], 5), 30), (([30, 11, 23, 4, 20], 6), 23)]
for (piles, h), want in examples:
    print(f"piles = {piles}, h = {h} -> {want}")

piles = [3, 6, 7, 11], h = 8 -> 4
piles = [30, 11, 23, 4, 20], h = 5 -> 30
piles = [30, 11, 23, 4, 20], h = 6 -> 23


## 3. Kiến thức nền

### 3.1 Làm tròn lên bằng số nguyên

Số giờ để ăn hết đống `p` ở tốc độ `k` là `⌈p / k⌉`. Python không có hàm làm tròn lên cho số nguyên, nhưng có hai mẹo **không dùng số thực**: `(p + k - 1) // k` và `-(-p // k)` (vì `//` làm tròn về `-∞`).

In [2]:
for p, k in [(7, 3), (6, 3), (1, 5), (11, 4)]:
    print(f"p={p}, k={k}: p//k={p // k}, ceil={(p + k - 1) // k} = {-(-p // k)} = math.ceil: {math.ceil(p / k)}")

p=7, k=3: p//k=2, ceil=3 = 3 = math.ceil: 3
p=6, k=3: p//k=2, ceil=2 = 2 = math.ceil: 2
p=1, k=5: p//k=0, ceil=1 = 1 = math.ceil: 1
p=11, k=4: p//k=2, ceil=3 = 3 = math.ceil: 3


### 3.2 Hàm số giờ

`hours(k) = Σ ⌈p / k⌉` với mọi đống. Khi `k` tăng, mỗi số hạng **không tăng** (chia cho số lớn hơn). Vậy `hours` là hàm **không tăng** theo `k`.

In [3]:
def hours(piles, k):
    return sum((p + k - 1) // k for p in piles)


piles = [3, 6, 7, 11]
print([(k, hours(piles, k)) for k in range(1, 13)])

[(1, 27), (2, 15), (3, 10), (4, 8), (5, 8), (6, 6), (7, 5), (8, 5), (9, 5), (10, 5), (11, 4), (12, 4)]


### 3.3 Tìm kiếm nhị phân trên đáp án

Khi có mệnh đề đơn điệu `ok(k) = (hours(k) ≤ h)`: `False` ở các `k` nhỏ rồi `True` ở các `k` lớn, ta tìm `k` nhỏ nhất có `ok(k) = True`. Với khoảng `[lo, hi]` chứa đáp án: `mid = (lo + hi) // 2`; nếu `ok(mid)` thì đáp án `≤ mid` → `hi = mid`; ngược lại `lo = mid + 1`. Dừng khi `lo == hi`.

### 3.4 `bisect_left` với `key`

Từ Python 3.10, `bisect_left(a, x, key=f)` tìm trong dãy `a` (đã "sắp theo `f`") vị trí đầu tiên mà `f(a[i]) ≥ x`. Với `a = range(1, max+1)` (không tốn bộ nhớ) và `f = lambda k: hours(k) <= h` (kiểu bool, xem `False < True`), `bisect_left(a, True, key=f)` trả chỉ số đầu tiên có `True`.

In [4]:
a = range(1, 12)
idx = bisect_left(a, True, key=lambda k: hours(piles, k) <= 8)
print("chỉ số đầu tiên có True:", idx, "-> k =", a[idx])

chỉ số đầu tiên có True: 3 -> k = 4


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, `h = 8`:

| `k` | giờ từng đống `[3,6,7,11]` | tổng |
|---|---|---|
| 3 | 1, 2, 3, 4 | 10 > 8 |
| 4 | 1, 2, 2, 3 | 8 ≤ 8 ✓ |
| 5 | 1, 2, 2, 3 | 8 ✓ |

Tốc độ càng cao thì càng nhanh xong: nếu `4` đủ thì `5, 6, …` cũng đủ. Vậy đáp án là ranh giới giữa "không đủ" và "đủ". Cận: `k = max(piles)` luôn đủ (mỗi đống một giờ, tổng `n ≤ h` giờ); `k = 1` là nhỏ nhất có thể.

In [5]:
for k in range(1, 8):
    per = [(p + k - 1) // k for p in piles]
    print(f"k = {k}: giờ từng đống {per}, tổng {sum(per)}, {'đủ' if sum(per) <= 8 else 'không đủ'}")

k = 1: giờ từng đống [3, 6, 7, 11], tổng 27, không đủ
k = 2: giờ từng đống [2, 3, 4, 6], tổng 15, không đủ
k = 3: giờ từng đống [1, 2, 3, 4], tổng 10, không đủ
k = 4: giờ từng đống [1, 2, 2, 3], tổng 8, đủ
k = 5: giờ từng đống [1, 2, 2, 3], tổng 8, đủ
k = 6: giờ từng đống [1, 1, 2, 2], tổng 6, đủ
k = 7: giờ từng đống [1, 1, 1, 2], tổng 5, đủ


## 5. Cách 1 — Brute Force: quét tuyến tính `k`

**Ý tưởng:** thử `k = 1, 2, 3, …` và dừng ở `k` đầu tiên có `hours(k) ≤ h`. Đúng vì `k` tăng dần nên `k` đầu tiên thoả là nhỏ nhất.

In [6]:
def eating_linear(piles, h):
    k = 1
    while hours(piles, k) > h:
        k += 1
    return k


for (p, h), want in examples:
    assert eating_linear(p, h) == want
print([eating_linear(p, h) for (p, h), _ in examples])

[4, 30, 23]


In [7]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng TLE khi đáp án ~ 10^9)
class SolutionLinear:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        k = 1
        while sum((p + k - 1) // k for p in piles) > h:
            k += 1
        return k

**Độ phức tạp:** tối đa `max(piles)` lần thử, mỗi lần `O(n)` → `O(n · max)` = `10^4 · 10^9`: quá chậm. Bộ nhớ `O(1)`.

**Nút thắt:** ta thử từng `k` trong khi tính **đơn điệu** cho phép loại một nửa khoảng mỗi bước.

## 6. Ý tưởng thuật toán / bước đột phá

`ok(k) = hours(k) ≤ h` là mệnh đề đơn điệu (`False…False True…True`). Tìm nhị phân `k` nhỏ nhất có `ok(k)`:

- khoảng `[1, max(piles)]` chứa đáp án;
- `O(log max)` bước, mỗi bước `O(n)` để tính `hours(mid)`.

Tổng `O(n log max)` ≈ `10^4 · 30 = 3·10^5` phép.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** nhị phân trên `[1, max]`, **Cách 3** nhị phân với cận dưới chặt, **Cách 4** `bisect_left` với `key`.

### Cách 2 — Nhị phân trên `[1, max(piles)]`

**Giải thích từng đoạn:**

1. `lo = 1`, `hi = max(piles)`.
2. Khi `lo < hi`: `mid = (lo + hi) // 2`; nếu `hours(mid) ≤ h` thì `hi = mid` (giữ `mid` làm ứng viên) ngược lại `lo = mid + 1`.
3. Khi `lo == hi` đó là đáp án.

In [8]:
def eating_bs(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) <= h:
            hi = mid                                     # mid đủ nhanh: đáp án <= mid
        else:
            lo = mid + 1                                 # mid chậm quá: đáp án > mid
    return lo


for (p, h), want in examples:
    assert eating_bs(p, h) == want
print([eating_bs(p, h) for (p, h), _ in examples])

[4, 30, 23]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionBS:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        lo, hi = 1, max(piles)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((p + mid - 1) // mid for p in piles) <= h:
                hi = mid
            else:
                lo = mid + 1
        return lo

**Độ phức tạp:** `O(n log M)` với `M = max(piles)`; bộ nhớ `O(1)`.

### Cách 3 — Nhị phân với cận dưới chặt

**Ý tưởng:** mỗi giờ ăn tối đa `k` quả nên cần ít nhất `⌈tổng / k⌉` giờ ⇒ để xong trong `h` giờ phải có `k ≥ ⌈tổng / h⌉`. Đặt `lo = max(1, ⌈tổng / h⌉)`. Cận trên vẫn `max(piles)`. Khoảng tìm nhỏ hơn, thường ít bước hơn.

In [10]:
def eating_bs_tight(piles, h):
    total = sum(piles)
    lo = max(1, (total + h - 1) // h)                    # mỗi giờ ăn <= k quả nên k >= ceil(tổng / h)
    hi = max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


for (p, h), want in examples:
    assert eating_bs_tight(p, h) == want
print([eating_bs_tight(p, h) for (p, h), _ in examples])

[4, 30, 23]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionTight:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        lo, hi = max(1, (sum(piles) + h - 1) // h), max(piles)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((p + mid - 1) // mid for p in piles) <= h:
                hi = mid
            else:
                lo = mid + 1
        return lo

**Độ phức tạp:** `O(n log(M − ⌈S/h⌉))`, tối đa như Cách 2. Cận dưới hợp lệ: `lo ≤` đáp án thật (xem mục 10).

### Cách 4 — `bisect_left` với `key`

**Ý tưởng:** dãy ảo `range(1, max + 1)`, khoá `ok(k)` kiểu `bool`; `bisect_left` tìm vị trí `True` đầu tiên. Gọn nhưng cần Python ≥ 3.10 (LeetCode hiện dùng bản đủ mới).

In [12]:
def eating_bisect(piles, h):
    ks = range(1, max(piles) + 1)                        # không cấp phát: range là dãy ảo
    return ks[bisect_left(ks, True, key=lambda k: hours(piles, k) <= h)]


for (p, h), want in examples:
    assert eating_bisect(p, h) == want
print([eating_bisect(p, h) for (p, h), _ in examples])

[4, 30, 23]


In [13]:
# Code nộp LeetCode cho Cách 4
from bisect import bisect_left


class SolutionBisect:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        ks = range(1, max(piles) + 1)
        return ks[bisect_left(ks, True, key=lambda k: sum((p + k - 1) // k for p in piles) <= h)]

**Độ phức tạp:** `O(n log M)`; bộ nhớ `O(1)`.

## 8. TRACE từng bước

**Trace Cách 2** với `piles = [3,6,7,11]`, `h = 8`: `lo = 1`, `hi = 11`.

| bước | `lo` | `hi` | `mid` | giờ từng đống | tổng | so với `h = 8` | cập nhật |
|---|---|---|---|---|---|---|---|
| 1 | 1 | 11 | 6 | 1,1,2,2 | 6 | ≤ | `hi = 6` |
| 2 | 1 | 6 | 3 | 1,2,3,4 | 10 | > | `lo = 4` |
| 3 | 4 | 6 | 5 | 1,2,2,3 | 8 | ≤ | `hi = 5` |
| 4 | 4 | 5 | 4 | 1,2,2,3 | 8 | ≤ | `hi = 4` |

`lo == hi == 4` → đáp án `4` ✓. Hàm trace in đúng bảng này:

In [14]:
def trace_bs(piles, h):
    lo, hi = 1, max(piles)
    print(f"piles = {piles}, h = {h}")
    print(f"{'lo':>3} {'hi':>3} {'mid':>4} {'giờ từng đống':<22} {'tổng':>5}  quyết định")
    while lo < hi:
        mid = (lo + hi) // 2
        per = [(p + mid - 1) // mid for p in piles]
        ok = sum(per) <= h
        action = f"hi = {mid}" if ok else f"lo = {mid + 1}"
        print(f"{lo:>3} {hi:>3} {mid:>4} {str(per):<22} {sum(per):>5}  {'<= h' if ok else '> h'} -> {action}")
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print("=> k =", lo)
    return lo


assert trace_bs([3, 6, 7, 11], 8) == 4
print()
assert trace_bs([30, 11, 23, 4, 20], 6) == 23

piles = [3, 6, 7, 11], h = 8
 lo  hi  mid giờ từng đống           tổng  quyết định
  1  11    6 [1, 1, 2, 2]               6  <= h -> hi = 6
  1   6    3 [1, 2, 3, 4]              10  > h -> lo = 4
  4   6    5 [1, 2, 2, 3]               8  <= h -> hi = 5
  4   5    4 [1, 2, 2, 3]               8  <= h -> hi = 4
=> k = 4

piles = [30, 11, 23, 4, 20], h = 6
 lo  hi  mid giờ từng đống           tổng  quyết định
  1  30   15 [2, 1, 2, 1, 2]            8  > h -> lo = 16
 16  30   23 [2, 1, 1, 1, 1]            6  <= h -> hi = 23
 16  23   19 [2, 1, 2, 1, 2]            8  > h -> lo = 20
 20  23   21 [2, 1, 2, 1, 1]            7  > h -> lo = 22
 22  23   22 [2, 1, 2, 1, 1]            7  > h -> lo = 23
=> k = 23


## 9. Code nộp LeetCode

Chọn **Cách 3** (cận dưới chặt). Giữ nguyên `class Solution`, `minEatingSpeed(self, piles, h)`:

In [15]:
from typing import List


class Solution:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        lo = max(1, (sum(piles) + h - 1) // h)            # k >= ceil(tổng / h)
        hi = max(piles)                                   # k = max luôn đủ vì n <= h
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((p + mid - 1) // mid for p in piles) <= h:   # mid đủ nhanh
                hi = mid
            else:
                lo = mid + 1
        return lo


print([Solution().minEatingSpeed(p, h) for (p, h), _ in examples])      # [4, 30, 23]
assert all(Solution().minEatingSpeed(p, h) == want for (p, h), want in examples)

[4, 30, 23]


## 10. Vì sao đúng

**(a) Tính đơn điệu.** Với `k1 < k2`: `⌈p/k1⌉ ≥ ⌈p/k2⌉` cho mọi `p`, nên `hours(k1) ≥ hours(k2)`. Nếu `hours(k1) ≤ h` thì `hours(k2) ≤ h`. Vậy tập `k` thoả mãn là một **đoạn đuôi** `[k*, ∞)`; đáp án là `k*`.

**(b) Công thức giờ.** Với tốc độ `k`, đống `p` cần đúng `⌈p/k⌉` giờ (mỗi giờ ăn tối đa `k` quả từ một đống, và phần giờ dư không dùng cho đống khác). Các đống độc lập nên tổng giờ tối thiểu là `Σ⌈p/k⌉`. (Thứ tự chọn đống không ảnh hưởng.)

**(c) Cận hợp lệ.** `k = max(piles)` cho mỗi đống đúng `1` giờ → `n ≤ h` giờ: thoả. Cận dưới: tổng số quả `S ≤ k · (số giờ)` ≤ `k · h`, nên `k ≥ ⌈S/h⌉`. Vậy `k*` nằm trong `[max(1, ⌈S/h⌉), max(piles)]`.

**(d) Vòng nhị phân.** Bất biến: `lo ≤ k* ≤ hi`. Nếu `ok(mid)` thì `k* ≤ mid` → `hi = mid` giữ bất biến; nếu không thì `k* > mid` → `lo = mid + 1`. Khoảng `hi − lo` giảm ít nhất một mỗi vòng (vì `mid < hi` khi `lo < hi`), nên kết thúc; khi `lo == hi`, `lo = k*`.

**(e) Bisect.** `bisect_left` với khoá bool cho chỉ số đầu tiên có `True`: chính là `k*`.

Ta kiểm chứng bằng máy: mọi cách khớp **oracle mô phỏng từng giờ** trên mọi dữ liệu nhỏ (mục 12D) và kiểm tra tính tối thiểu `ok(k*) ∧ ¬ok(k*−1)` trên dữ liệu lớn (12E–F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quét tuyến tính | `O(n · M)` | `O(1)` | oracle một phần; TLE |
| 2. Nhị phân `[1, M]` | `O(n log M)` | `O(1)` | chuẩn |
| 3. Nhị phân cận chặt | `O(n log M)` | `O(1)` | **đáp án chính** |
| 4. `bisect_left` + `key` | `O(n log M)` | `O(1)` | gọn, Python ≥ 3.10 |

Đo thực tế (`n = 10^4`, giá trị tới `10^9`; cách 1 chỉ với giá trị nhỏ):

In [16]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(875)
big = [rng.randint(1, 10 ** 9) for _ in range(10 ** 4)]
hh = rng.randint(10 ** 4, 10 ** 5)
for name, fn in [("bs", eating_bs), ("tight", eating_bs_tight), ("bisect", eating_bisect)]:
    print(f"n = 10^4, M = 10^9 {name:7s} {bench(fn, (big, hh)) * 1e3:8.2f} ms")
small = [rng.randint(1, 3000) for _ in range(300)]
print(f"cách 1 (giá trị <= 3000, n = 300): {bench(eating_linear, (small, 301), 1) * 1e3:8.1f} ms (tăng tuyến tính theo đáp án)")

n = 10^4, M = 10^9 bs         19.95 ms


n = 10^4, M = 10^9 tight      29.99 ms


n = 10^4, M = 10^9 bisect     28.16 ms


cách 1 (giá trị <= 3000, n = 300):     71.9 ms (tăng tuyến tính theo đáp án)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi dữ liệu nhỏ với oracle **mô phỏng từng giờ**, (E) ngẫu nhiên có seed và kiểm tra tính tối thiểu, (F) giới hạn.

### A. Ví dụ của đề

In [17]:
def simulate_hours(piles, k):
    """Mô phỏng ĐÚNG đề: mỗi giờ chọn một đống còn chuối, ăn min(k, còn lại). Trả về số giờ cần."""
    rem = list(piles)
    hrs = 0
    i = 0
    while i < len(rem):
        if rem[i] == 0:
            i += 1
            continue
        rem[i] -= min(k, rem[i])                         # một giờ: ăn từ một đống, phần giờ dư bị bỏ
        hrs += 1
    return hrs


def oracle(piles, h):
    k = 1
    while simulate_hours(piles, k) > h:                  # mô phỏng từng giờ rồi quét k = 1, 2, ...
        k += 1
    return k


IMPLS = {
    "linear": eating_linear,
    "bs": eating_bs,
    "tight": eating_bs_tight,
    "bisect": eating_bisect,
    "Solution": lambda p, h: Solution().minEatingSpeed(list(p), h),
    "SolutionLinear": lambda p, h: SolutionLinear().minEatingSpeed(list(p), h),
    "SolutionBS": lambda p, h: SolutionBS().minEatingSpeed(list(p), h),
    "SolutionTight": lambda p, h: SolutionTight().minEatingSpeed(list(p), h),
    "SolutionBisect": lambda p, h: SolutionBisect().minEatingSpeed(list(p), h),
}


def check(piles, h, want=None, names=None):
    if want is None:
        want = oracle(piles, h)
    for name in (names or IMPLS):
        got = IMPLS[name](piles, h)
        assert got == want, (name, piles, h, got, want)


for (p, h), want in examples:
    check(p, h, want)
    assert oracle(p, h) == want
print("ví dụ đề: oracle mô phỏng từng giờ và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle mô phỏng từng giờ và 9 cài đặt đúng


### B. Edge case

- Một đống: `piles = [n]` với `h`: `k = ⌈n / h⌉`.
- `h = n` (mỗi đống đúng một giờ): `k = max(piles)`.
- `h` rất lớn (`h ≥ Σ piles`): `k = 1`.
- Mọi đống bằng nhau.
- Đống `1`.
- `h` vừa đủ làm cận dưới `⌈S/h⌉` **không** phải đáp án (đống lệch nhau).

In [18]:
cases = [
    ([1], 1, 1), ([10], 1, 10), ([10], 3, 4), ([10], 10, 1), ([10], 100, 1),
    ([1, 1, 1, 1], 4, 1), ([5, 5, 5], 3, 5), ([5, 5, 5], 6, 3), ([5, 5, 5], 15, 1),
    ([30, 11, 23, 4, 20], 5, 30), ([100, 1, 1, 1], 5, None),
    ([2, 2], 2, 2), ([1000000000], 2, 500000000), ([1000000000, 1000000000], 3, 1000000000),
]
for p, h, want in cases:
    check(p, h, want, names=["bs", "tight", "bisect", "Solution", "SolutionBS", "SolutionTight", "SolutionBisect"])
for p, h, want in cases[:11]:
    check(p, h, want)                                    # cả cách quét tuyến tính (giá trị nhỏ)
print(len(cases), "edge case: đúng")

14 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_floor(piles, h):                               # giờ = p // k (làm tròn xuống)
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(p // mid for p in piles) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_total_over_k(piles, h):                        # giờ = ceil(tổng / k): quên mỗi giờ chỉ ăn một đống
    S = sum(piles)
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if (S + mid - 1) // mid <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_strict(piles, h):                              # hours < h thay vì <=
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) < h:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_hi_exclusive(piles, h):                        # cận trên max - 1: loại mất đáp án = max
    lo, hi = 1, max(piles) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_lo_mid_plus(piles, h):                         # hi = mid - 1 rồi trả lo: bỏ mất ứng viên mid
    lo, hi = 1, max(piles)
    while lo <= hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) <= h:
            hi = mid - 1
        else:
            lo = mid + 1
    return lo - 1                                      # SAI: nên trả lo (không trừ 1)


BUGS = [bug_floor, bug_total_over_k, bug_strict, bug_hi_exclusive, bug_lo_mid_plus]
rng = random.Random(3)
tests = []
for _ in range(600):
    n = rng.randint(1, 5)
    p = [rng.randint(1, 12) for _ in range(n)]
    tests.append((p, rng.randint(n, sum(p) + 1)))
for bug in BUGS:
    bad = next(((p, h) for p, h in tests if bug(p, h) != Solution().minEatingSpeed(p, h)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên piles = {bad[0]}, h = {bad[1]}: ra {bug(*bad)}, đúng {Solution().minEatingSpeed(*bad)}")

bug_floor          sai trên piles = [10, 9], h = 6: ra 3, đúng 4
bug_total_over_k   sai trên piles = [9, 8, 7, 11, 3], h = 19: ra 2, đúng 3
bug_strict         sai trên piles = [10, 9], h = 6: ra 5, đúng 4
bug_hi_exclusive   sai trên piles = [10], h = 1: ra 9, đúng 10
bug_lo_mid_plus    sai trên piles = [10, 9], h = 6: ra 3, đúng 4


### D. Vét cạn mọi dữ liệu nhỏ

Mọi `piles` có độ dài `1..4`, giá trị `1..6`, và **mọi** `h` từ `n` tới `Σ piles` (với `h > Σ` đáp án vẫn `1`, thêm `h = Σ + 1` để kiểm). Oracle: mô phỏng từng giờ.

In [20]:
import itertools

t0 = time.perf_counter()
cnt = 0
fast = ("linear", "bs", "tight", "bisect", "Solution")
for n in range(1, 5):
    for p in itertools.product(range(1, 7), repeat=n):
        p = list(p)
        for h in range(n, sum(p) + 2):
            want = oracle(p, h)
            for name in fast:
                assert IMPLS[name](p, h) == want, (name, p, h)
            cnt += 1
print(cnt, "bộ (piles, h): 5 cài đặt khớp oracle mô phỏng từng giờ;", round(time.perf_counter() - t0, 1), "s")

17883 bộ (piles, h): 5 cài đặt khớp oracle mô phỏng từng giờ; 0.2 s


### E. Ngẫu nhiên có seed và tính tối thiểu

Dữ liệu ngẫu nhiên cỡ vừa so với oracle; và trên dữ liệu **lớn** kiểm tra trực tiếp định nghĩa: `hours(k*) ≤ h` và `hours(k* − 1) > h` (hoặc `k* = 1`).

In [21]:
rng = random.Random(20240875)
for _ in range(1500):
    n = rng.randint(1, 8)
    p = [rng.randint(1, 40) for _ in range(n)]
    h = rng.randint(n, sum(p) + 2)
    check(p, h)
print("1500 ca ngẫu nhiên nhỏ: 9 cài đặt khớp oracle")

for _ in range(300):
    n = rng.randint(1, 2000)
    p = [rng.randint(1, rng.choice([10, 1000, 10 ** 9])) for _ in range(n)]
    h = rng.randint(n, rng.choice([n + 5, 3 * n, 10 ** 6, 10 ** 9]))
    k = Solution().minEatingSpeed(p, h)
    assert hours(p, k) <= h and (k == 1 or hours(p, k - 1) > h)            # tối thiểu theo định nghĩa
    assert eating_bs(p, h) == eating_bs_tight(p, h) == eating_bisect(p, h) == k
print("300 ca lớn: nghiệm thoả ok(k) và không ok(k-1); 4 cách khớp nhau")

1500 ca ngẫu nhiên nhỏ: 9 cài đặt khớp oracle


300 ca lớn: nghiệm thoả ok(k) và không ok(k-1); 4 cách khớp nhau


### F. Giới hạn

`n = 10^4`, giá trị `10^9`, `h` từ `n` tới `10^9`; và các trường hợp biên: `h = n` ⇒ `k = max`; `h = 10^9` với đống nhỏ ⇒ `1`.

In [22]:
rng = random.Random(777)
n = 10 ** 4
big = [rng.randint(1, 10 ** 9) for _ in range(n)]
assert Solution().minEatingSpeed(big, n) == max(big)
for h in (n, n + 1, 2 * n, 10 ** 5, 10 ** 7, 10 ** 9):
    k = Solution().minEatingSpeed(big, h)
    assert hours(big, k) <= h and (k == 1 or hours(big, k - 1) > h)
    assert eating_bs(big, h) == eating_bisect(big, h) == k
    print(f"h = {h:>10}: k = {k}")
assert Solution().minEatingSpeed([1] * n, n) == 1
assert Solution().minEatingSpeed([10 ** 9] * n, n) == 10 ** 9
print("giới hạn: đúng")

h =      10000: k = 999870107


h =      10001: k = 999781069


h =      20000: k = 334938973


h =     100000: k = 52675570


h =   10000000: k = 500517


h = 1000000000: k = 5003
giới hạn: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: làm tròn xuống thay vì lên

Đống `7` quả ở tốc độ `3` cần `3` giờ (`3 + 3 + 1`), không phải `2`.

In [23]:
print("p=7,k=3: p//k =", 7 // 3, "| đúng ceil =", (7 + 2) // 3)
p, h = [3, 6, 7, 11], 8
print("floor:", bug_floor(p, h), "| đúng:", Solution().minEatingSpeed(p, h))
assert bug_floor(p, h) != Solution().minEatingSpeed(p, h)

p=7,k=3: p//k = 2 | đúng ceil = 3
floor: 3 | đúng: 4


**Vì sao sai:** phần dư vẫn tốn một giờ. **Đúng:** `(p + k - 1) // k`.

### Lỗi 2: tính giờ bằng tổng chia tốc độ

Quên rằng mỗi giờ chỉ ăn **một đống**; phần giờ dư của đống nhỏ bị bỏ phí.

In [24]:
p, h = [30, 11, 23, 4, 20], 5
print("⌈tổng/k⌉:", bug_total_over_k(p, h), "| đúng:", Solution().minEatingSpeed(p, h))
assert bug_total_over_k(p, h) < Solution().minEatingSpeed(p, h)

⌈tổng/k⌉: 18 | đúng: 30


**Vì sao sai:** tổng `88`, nhưng mỗi đống cần ít nhất một giờ riêng. **Đúng:** cộng `⌈pile/k⌉` **từng đống**.

### Lỗi 3: `<` thay vì `≤`

Đề cho phép dùng **đủ** `h` giờ.

In [25]:
p, h = [3, 6, 7, 11], 8
print("strict:", bug_strict(p, h), "| đúng:", Solution().minEatingSpeed(p, h))
assert bug_strict(p, h) == 6

strict: 6 | đúng: 4


**Vì sao sai:** `k = 4` dùng đúng `8` giờ là hợp lệ. **Đúng:** `hours(k) <= h`.

### Lỗi 4: cận trên loại mất đáp án

`hi = max - 1` sai khi đáp án chính là `max(piles)` (ví dụ 2).

In [26]:
p, h = [30, 11, 23, 4, 20], 5
print("hi = max-1:", bug_hi_exclusive(p, h), "| đúng:", Solution().minEatingSpeed(p, h))
assert bug_hi_exclusive(p, h) != 30

hi = max-1: 29 | đúng: 30


**Vì sao sai:** `k = max` là cận trên **hợp lệ**, phải nằm trong khoảng. **Đúng:** `hi = max(piles)`.

### Lỗi 5: vòng nhị phân lệch một

Dùng `lo <= hi` và `hi = mid - 1` rồi trả `lo - 1`: dễ nhầm.

In [27]:
p, h = [3, 6, 7, 11], 8
print("lệch một:", bug_lo_mid_plus(p, h), "| đúng:", Solution().minEatingSpeed(p, h))
assert bug_lo_mid_plus(p, h) != Solution().minEatingSpeed(p, h)

lệch một: 3 | đúng: 4


**Vì sao sai:** với khung `lo <= hi` + `hi = mid - 1` thì đáp án là `lo` (không trừ). **Đúng:** thống nhất một khung, ví dụ `lo < hi`, `hi = mid`, trả `lo`.

### Lỗi 6 (bonus): `lo = 0`

`mid` có thể bằng `0` ⇒ chia cho `0`. Cận dưới phải là `1`.

In [28]:
try:
    hours([3, 6], 0)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

ZeroDivisionError: integer division or modulo by zero


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tìm giá trị **nhỏ nhất** thoả một điều kiện đơn điệu theo giá trị đó" (tốc độ tối thiểu, dung lượng tối thiểu, thời gian tối thiểu) → **nhị phân trên đáp án**.

**Mẹo:**

1. Viết hàm `ok(k)` và kiểm tra tính đơn điệu trước.
2. Chọn cận: dưới là giá trị nhỏ nhất có thể (kể cả cận chặt `⌈S/h⌉`), trên là một giá trị **chắc chắn đủ**.
3. Khung chuẩn: `while lo < hi: mid = (lo+hi)//2; if ok(mid): hi = mid else: lo = mid+1; return lo`.
4. Làm tròn lên bằng số nguyên: `(a + b - 1) // b`.
5. Oracle: mô phỏng đúng đề, quét tuyến tính `k`.

**Họ bài cùng pattern:** Capacity To Ship Packages Within D Days (1011), Split Array Largest Sum (410), Minimum Time to Complete Trips (2187), Maximum Candies Allocated to K Children (2226), Magnetic Force Between Two Balls (1552), Find the Smallest Divisor Given a Threshold (1283).

## 15. Biến thể

1. **Ship capacity (1011):** các gói **theo thứ tự**, mỗi ngày chở liên tiếp tối đa `cap`; min `cap` để xong trong `D` ngày.
2. **Smallest divisor (1283):** `Σ⌈a/d⌉ ≤ threshold`.
3. **Mỗi giờ có thể ăn nhiều đống** (chuối dư chuyển sang đống kế): đáp án `⌈S/h⌉`.
4. **Số giờ tối thiểu cho tốc độ cố định** (`hours(k)`).
5. **Nhiều Koko** chia đống.

Cài biến thể 1, 2 và 3 và đối chiếu vét cạn:

In [29]:
def ship_days(weights, cap):
    days, cur = 1, 0
    for w in weights:
        if cur + w > cap:
            days += 1
            cur = 0
        cur += w
    return days


def ship_capacity(weights, D):
    lo, hi = max(weights), sum(weights)                  # tối thiểu phải chở được gói nặng nhất
    while lo < hi:
        mid = (lo + hi) // 2
        if ship_days(weights, mid) <= D:
            hi = mid
        else:
            lo = mid + 1
    return lo


def ship_brute(weights, D):
    cap = max(weights)
    while ship_days(weights, cap) > D:
        cap += 1
    return cap


def smallest_divisor(nums, threshold):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo


def divisor_brute(nums, threshold):
    d = 1
    while sum((x + d - 1) // d for x in nums) > threshold:
        d += 1
    return d


def pooled_speed(piles, h):
    """Biến thể 3: bỏ phí giờ dư = không còn: mỗi giờ ăn đủ k quả (có thể sang đống khác) -> k = ceil(S/h)."""
    return max(1, (sum(piles) + h - 1) // h)


rng = random.Random(15)
for _ in range(400):
    w = [rng.randint(1, 15) for _ in range(rng.randint(1, 8))]
    D = rng.randint(1, len(w))
    assert ship_capacity(w, D) == ship_brute(w, D)
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    th = rng.randint(len(nums), sum(nums))
    assert smallest_divisor(nums, th) == divisor_brute(nums, th)
    p = [rng.randint(1, 12) for _ in range(rng.randint(1, 5))]
    h = rng.randint(1, sum(p))
    # mỗi giờ ăn đủ k quả bất kể đống: số giờ = ceil(S / k); k nhỏ nhất với ceil(S/k) <= h
    k = pooled_speed(p, h)
    assert (sum(p) + k - 1) // k <= h and (k == 1 or (sum(p) + k - 2) // (k - 1) > h)
print("ship capacity (1011), smallest divisor (1283), tốc độ gộp: khớp vét cạn/định nghĩa trên 400 ca")
assert ship_capacity([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15 and smallest_divisor([1, 2, 5, 9], 6) == 5

ship capacity (1011), smallest divisor (1283), tốc độ gộp: khớp vét cạn/định nghĩa trên 400 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tuyến tính | thử `k = 1, 2, …` | `O(n·M)` | `O(1)` | oracle, `M` nhỏ |
| 2. Nhị phân `[1, M]` | loại nửa khoảng nhờ đơn điệu | `O(n log M)` | `O(1)` | chuẩn |
| 3. Nhị phân cận chặt | `lo = ⌈S/h⌉` | `O(n log M)` | `O(1)` | **đáp án chính** |
| 4. `bisect_left` + `key` | tìm `True` đầu tiên | `O(n log M)` | `O(1)` | gọn gàng |

Cả bốn khớp oracle mô phỏng từng giờ trên mọi dữ liệu nhỏ (mục 12D) và khớp nhau, kèm kiểm tra tính tối thiểu, trên dữ liệu lớn.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().minEatingSpeed([5, 5, 5], 3), Solution().minEatingSpeed([5, 5, 5], 6), Solution().minEatingSpeed([10], 3), hours([3, 6, 7, 11], 3))

5 3 4 10


### Bài 2 (tính tay)

`piles = [4, 9, 2]`, `h = 5`. (a) Tính `hours(k)` cho `k = 1..9`. (b) Tìm `k` nhỏ nhất. (c) Cận dưới `⌈S/h⌉` bằng bao nhiêu và có là đáp án không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện nhị phân:

In [31]:
def bs_todo(piles, h):
    lo, hi = 0, 0       # TODO 1: cận dưới, cận trên hợp lệ
    while lo < hi:
        mid = (lo + hi) // 2
        pass            # TODO 2: nếu hours(mid) <= h thì hi = mid, ngược lại lo = mid + 1
    return lo


# Khi làm xong, bỏ comment để kiểm tra:
# assert [bs_todo(p, h) for (p, h), _ in examples] == [4, 30, 23]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`bisect_left` với `key`**:

In [32]:
class MySolution:
    def minEatingSpeed(self, piles, h):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().minEatingSpeed(p, h) == w for (p, h), w in examples)

### Bài 5 (mini challenge): giờ tối thiểu cho tốc độ cố định và ngược lại

Viết `max_speed_needed(piles)` = tốc độ **lớn nhất** có ích (tăng thêm không làm giảm giờ nữa) và `min_hours(piles)` = số giờ nhỏ nhất có thể (với tốc độ vô hạn). Kiểm tra: với `k = max_speed_needed`, `hours(k) = min_hours`, và với `k − 1` thì `hours` lớn hơn (nếu `k > 1`).

In [33]:
def max_speed_needed(piles):
    pass            # TODO


def min_hours(piles):
    pass            # TODO


# Khi làm xong:
# p = [3, 6, 7, 11]; assert max_speed_needed(p) == 11 and min_hours(p) == 4

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tính từng đống `⌈p/k⌉`.
- Mức 2: `[5,5,5], h=3` mỗi đống một giờ → `5`; `h=6` mỗi đống hai giờ → `3`; `[10], h=3` → `⌈10/3⌉ = 4`.
- Mức 3: `5 3 4 10` (giờ ở `k=3`: `1+2+3+4 = 10`).

**Bài 2**

- Mức 1: `hours(k) = ⌈4/k⌉ + ⌈9/k⌉ + ⌈2/k⌉`.
- Mức 2: `hours(3) = 2+3+1 = 6 > 5`; `hours(4) = 1+3+1 = 5`.
- Mức 3: `k = 4`; cận dưới `⌈15/5⌉ = 3` **không** đủ → không phải đáp án.

**Bài 3**

- Mức 1: `lo = 1`, `hi = max(piles)`.
- Mức 2: `hours(piles, mid) <= h` → `hi = mid`.
- Mức 3: trả `lo`.

**Bài 4**

- Mức 1: dãy ảo `range(1, max + 1)`.
- Mức 2: khoá `lambda k: hours(piles, k) <= h`.
- Mức 3: `ks[bisect_left(ks, True, key=...)]`.

**Bài 5**

- Mức 1: `hours(k)` giảm tới khi `k ≥ max`.
- Mức 2: với `k ≥ max(piles)` mỗi đống một giờ.
- Mức 3: `max_speed_needed = max(piles)`, `min_hours = len(piles)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
assert (Solution().minEatingSpeed([5, 5, 5], 3), Solution().minEatingSpeed([5, 5, 5], 6), Solution().minEatingSpeed([10], 3), hours([3, 6, 7, 11], 3)) == (5, 3, 4, 10)
print("Bài 1 OK: 5 3 4 10")

Bài 1 OK: 5 3 4 10


### Bài 2

`hours(1..9) = 15, 8, 6, 5, 4, 4, 4, 4, 3`; với `h = 5` thì `k = 4` là nhỏ nhất (`hours(4) = 5`); cận dưới `⌈15/5⌉ = 3` không đủ vì `hours(3) = 6 > 5`.

In [35]:
p, h = [4, 9, 2], 5
table = [hours(p, k) for k in range(1, 10)]
print(table)
assert table == [15, 8, 6, 5, 4, 4, 4, 4, 3]
assert Solution().minEatingSpeed(p, h) == 4 == oracle(p, h)
assert (sum(p) + h - 1) // h == 3 and hours(p, 3) > h
print("Bài 2 OK")

[15, 8, 6, 5, 4, 4, 4, 4, 3]
Bài 2 OK


### Bài 3

In [36]:
def bs_answer(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(piles, mid) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert [bs_answer(p, h) for (p, h), _ in examples] == [4, 30, 23]
rng = random.Random(3)
for _ in range(300):
    p = [rng.randint(1, 20) for _ in range(rng.randint(1, 5))]
    h = rng.randint(len(p), sum(p))
    assert bs_answer(p, h) == oracle(p, h)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def minEatingSpeed(self, piles, h):
        ks = range(1, max(piles) + 1)
        return ks[bisect_left(ks, True, key=lambda k: sum((p + k - 1) // k for p in piles) <= h)]


assert all(MySolutionAnswer().minEatingSpeed(p, h) == w for (p, h), w in examples)
rng = random.Random(4)
for _ in range(300):
    p = [rng.randint(1, 20) for _ in range(rng.randint(1, 5))]
    h = rng.randint(len(p), sum(p))
    assert MySolutionAnswer().minEatingSpeed(p, h) == oracle(p, h)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def max_speed_needed_answer(piles):
    return max(piles)


def min_hours_answer(piles):
    return len(piles)


p = [3, 6, 7, 11]
assert max_speed_needed_answer(p) == 11 and min_hours_answer(p) == 4
rng = random.Random(5)
for _ in range(300):
    p = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    k = max_speed_needed_answer(p)
    assert hours(p, k) == min_hours_answer(p) == hours(p, k + 5)
    if k > 1:
        assert hours(p, k - 1) >= hours(p, k)
print("Bài 5 OK: tăng tốc quá max(piles) không giảm thêm giờ; số giờ nhỏ nhất = số đống")

Bài 5 OK: tăng tốc quá max(piles) không giảm thêm giờ; số giờ nhỏ nhất = số đống


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (nhị phân trên đáp án, tìm giá trị nhỏ nhất thoả điều kiện đơn điệu):**

```python
class Solution:
    def minEatingSpeed(self, piles, h):
        lo, hi = max(1, (sum(piles) + h - 1) // h), max(piles)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((p + mid - 1) // mid for p in piles) <= h:   # ok(mid)
                hi = mid
            else:
                lo = mid + 1
        return lo
```

**Checklist trước khi submit:**

- [ ] `ok(k)` đơn điệu (`False … False True … True`)?
- [ ] Làm tròn **lên** từng đống: `(p + k - 1) // k`, không `p // k`, không `⌈tổng/k⌉`?
- [ ] `≤ h`, không `< h`?
- [ ] Cận `lo ≥ 1` (không chia cho 0), `hi = max(piles)` (gồm cả đáp án `max`)?
- [ ] Khung nhất quán: `lo < hi`, `hi = mid`, `lo = mid + 1`, trả `lo`?
- [ ] Đã thử `[3,6,7,11],8 → 4`, `[30,11,23,4,20],5 → 30`, `...,6 → 23`, một đống, `h = n`?
- [ ] Đúng tên `class Solution`, `minEatingSpeed(self, piles, h)`, không in debug.